# Accent Neutralizer for Speech

A Deep Learning + Speech Processing project that transforms speech from non-native accents toward a native-English reference accent, while aiming to preserve the original sentence content.

**GSSoC 2026 Contribution — Issue #1956**

## Overview

This v1 implementation focuses on **acoustic feature conversion** (MFCC-level) as a baseline approach. Given a speech clip in a non-native accent (e.g., Hindi-accented or Chinese-accented English), the model learns to map its MFCC features toward the corresponding features of native-accent speech, using a BiLSTM-based sequence model trained on parallel accented/native sentence pairs.

**Scope note:** true accent conversion is an open research problem — accent and speaker identity are entangled in the same acoustic features (MFCCs, formants), so a model attempting to change *only* accent while perfectly preserving voice identity has to disentangle signals that overlap heavily. This v1 targets a scoped baseline (2 accent pairs) rather than a general-purpose, production-grade solution. "Neutral" here refers to a specific reference accent (General American, via the CMU ARCTIC corpus), not an absence of accent.

## Approach

1. **Data**: Parallel sentence pairs from [L2-ARCTIC](https://huggingface.co/datasets/KoelLabs/L2Arctic) (non-native) and [CMU ARCTIC](https://huggingface.co/datasets/MikhailT/cmu-arctic) (native reference), matched by sentence text → **1,148 aligned pairs** (575 Hindi, 573 Chinese)
2. **Feature extraction**: MFCC, pitch, and formants extracted per clip
3. **Alignment**: DTW alignment handles differing speaking rates between accented/native pairs
4. **Model**: BiLSTM encoder-decoder (~576K params) maps accented MFCCs toward native MFCCs
5. **Reconstruction**: Converted MFCCs turned back into audio via Griffin-Lim


## 1. Setup — Install Dependencies

In [ ]:
!pip install -q librosa numpy soundfile scikit-learn torch torchcodec scipy matplotlib datasets huggingface_hub tqdm nbformat

## 2. Feature Extractor

Extracts MFCC, pitch, and formant features from raw audio.

In [ ]:
"""
feature_extractor.py
Extracts acoustic features (MFCC, pitch, formants) from speech audio
for accent analysis and conversion.
"""

import numpy as np
import librosa


class FeatureExtractor:
    def __init__(self, sample_rate=16000, n_mfcc=13, n_fft=1024, hop_length=256):
        self.sr = sample_rate
        self.n_mfcc = n_mfcc
        self.n_fft = n_fft
        self.hop_length = hop_length

    def extract_mfcc(self, audio):
        """Extract MFCC features (timbre/phonetic content)."""
        mfcc = librosa.feature.mfcc(
            y=audio, sr=self.sr, n_mfcc=self.n_mfcc,
            n_fft=self.n_fft, hop_length=self.hop_length
        )
        return mfcc  # shape: (n_mfcc, time_frames)

    def extract_pitch(self, audio):
        """Extract fundamental frequency (F0) contour using pyin."""
        f0, voiced_flag, voiced_probs = librosa.pyin(
            audio, fmin=librosa.note_to_hz('C2'),
            fmax=librosa.note_to_hz('C7'),
            sr=self.sr, hop_length=self.hop_length
        )
        f0 = np.nan_to_num(f0)  # replace unvoiced NaNs with 0
        return f0

    def extract_formants(self, audio, n_formants=3, order=12):
        """
        Estimate formants (F1, F2, F3) using LPC analysis per frame.
        Formants carry most of the accent-distinguishing information.
        """
        frame_length = self.n_fft
        hop = self.hop_length
        formants_over_time = []

        for start in range(0, len(audio) - frame_length, hop):
            frame = audio[start:start + frame_length] * np.hamming(frame_length)
            try:
                lpc_coeffs = librosa.lpc(frame, order=order)
                roots = np.roots(lpc_coeffs)
                roots = roots[np.imag(roots) >= 0]
                angles = np.arctan2(np.imag(roots), np.real(roots))
                freqs = angles * (self.sr / (2 * np.pi))
                freqs = sorted(freqs[freqs > 90])  # discard near-zero/negative
                formants_over_time.append(freqs[:n_formants])
            except Exception:
                formants_over_time.append([0] * n_formants)

        return formants_over_time

    def extract_all(self, audio):
        """Convenience method: extract MFCC, pitch, and formants together."""
        return {
            'mfcc': self.extract_mfcc(audio),
            'pitch': self.extract_pitch(audio),
            'formants': self.extract_formants(audio),
        }


if __name__ == "__main__":
    # Quick smoke test with random noise (replace with real audio when integrating)
    dummy_audio = np.random.randn(16000 * 2).astype(np.float32)  # 2 sec of noise
    extractor = FeatureExtractor()
    features = extractor.extract_all(dummy_audio)
    print("MFCC shape:", features['mfcc'].shape)
    print("Pitch shape:", features['pitch'].shape)
    print("Number of formant frames:", len(features['formants']))
    print("Sample formants (first frame):", features['formants'][0])


## 3. Model Architecture

A lightweight BiLSTM encoder-decoder that learns to map accented MFCC features toward native-accent MFCC features.

In [ ]:
"""
neutralizer_model.py
A lightweight sequence model that learns to map accented MFCC features
toward native-accent MFCC features (accent neutralization).
"""

import torch
import torch.nn as nn


class AccentNeutralizer(nn.Module):
    def __init__(self, n_mfcc=13, hidden_dim=128, num_layers=2):
        super().__init__()
        self.n_mfcc = n_mfcc

        self.encoder = nn.LSTM(
            input_size=n_mfcc,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.2 if num_layers > 1 else 0.0
        )

        self.decoder = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, n_mfcc)
        )

    def forward(self, x):
        encoded, _ = self.encoder(x)
        output = self.decoder(encoded)
        return x + output


if __name__ == "__main__":
    model = AccentNeutralizer()
    dummy_input = torch.randn(4, 100, 13)
    output = model(dummy_input)
    print("Input shape:", dummy_input.shape)
    print("Output shape:", output.shape)
    num_params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters: {num_params:,}")


## 4. Data Preparation

Downloads L2-ARCTIC and CMU ARCTIC datasets, matches sentence pairs, aligns them using Dynamic Time Warping (DTW), and prepares training tensors.

In [ ]:
from datasets import load_dataset, Audio
import io
import soundfile as sf
import numpy as np
import pickle
import os
import re
from feature_extractor import FeatureExtractor
from tqdm import tqdm

def normalize_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("Loading datasets...")
l2 = load_dataset("KoelLabs/L2Arctic")
l2 = l2.cast_column("audio", Audio(decode=False))
cmu = load_dataset("MikhailT/cmu-arctic")
cmu = cmu.cast_column("audio", Audio(decode=False))

TARGET_LANGUAGES = ['Hindi', 'Chinese']
NATIVE_SPEAKERS = ['bdl', 'slt']

print("Building native sentence lookup...")
native_lookup = {}
for spk in NATIVE_SPEAKERS:
    for row in cmu[spk]:
        text_key = normalize_text(row['text'])
        if text_key not in native_lookup:
            native_lookup[text_key] = row

print("Matching pairs...")
pairs = []
for row in l2['scripted']:
    if row['speaker_native_language'] not in TARGET_LANGUAGES:
        continue
    text_key = normalize_text(row['text'])
    if text_key in native_lookup:
        pairs.append({'accented': row, 'native': native_lookup[text_key], 'language': row['speaker_native_language']})

print(f"Total pairs: {len(pairs)}")

extractor = FeatureExtractor()
processed_data = []

os.makedirs('data/processed', exist_ok=True)

print("Extracting features for all pairs (this will take a while)...")
for i, pair in enumerate(tqdm(pairs)):
    try:
        acc_bytes = pair['accented']['audio']['bytes']
        acc_audio, acc_sr = sf.read(io.BytesIO(acc_bytes))
        acc_audio = acc_audio.astype(np.float32)

        nat_bytes = pair['native']['audio']['bytes']
        nat_audio, nat_sr = sf.read(io.BytesIO(nat_bytes))
        nat_audio = nat_audio.astype(np.float32)

        acc_features = extractor.extract_all(acc_audio)
        nat_features = extractor.extract_all(nat_audio)

        processed_data.append({
            'text': pair['accented']['text'],
            'language': pair['language'],
            'accented_speaker': pair['accented']['speaker_code'],
            'native_speaker': pair['native']['speaker'],
            'accented_mfcc': acc_features['mfcc'],
            'accented_pitch': acc_features['pitch'],
            'native_mfcc': nat_features['mfcc'],
            'native_pitch': nat_features['pitch'],
        })
    except Exception as e:
        print(f"Skipping pair {i} due to error: {e}")
        continue

    if (i + 1) % 200 == 0:
        with open('data/processed/features_checkpoint.pkl', 'wb') as f:
            pickle.dump(processed_data, f)
        print(f"Checkpoint saved at {i+1} pairs")

with open('data/processed/features_final.pkl', 'wb') as f:
    pickle.dump(processed_data, f)

print(f"\nDone! Successfully processed {len(processed_data)} out of {len(pairs)} pairs")
print("Saved to data/processed/features_final.pkl")


In [ ]:
# See extract_all_features.py logic above — run as a script if needed:
# !python extract_all_features.py

In [ ]:
"""
prepare_training_data.py
Aligns accented/native MFCC pairs using DTW, then pads to a fixed
length so they can be batched for training.
"""

import pickle
import numpy as np
import librosa
from tqdm import tqdm

MAX_FRAMES = 300

def align_pair(acc_mfcc, nat_mfcc):
    D, wp = librosa.sequence.dtw(X=acc_mfcc, Y=nat_mfcc, metric='euclidean')
    wp = wp[::-1]

    aligned_acc = np.zeros_like(nat_mfcc)
    counts = np.zeros(nat_mfcc.shape[1])
    for acc_idx, nat_idx in wp:
        aligned_acc[:, nat_idx] += acc_mfcc[:, acc_idx]
        counts[nat_idx] += 1
    counts[counts == 0] = 1
    aligned_acc = aligned_acc / counts

    return aligned_acc

def pad_or_crop(mfcc, max_frames=MAX_FRAMES):
    n_frames = mfcc.shape[1]
    if n_frames >= max_frames:
        return mfcc[:, :max_frames]
    else:
        pad_width = max_frames - n_frames
        return np.pad(mfcc, ((0, 0), (0, pad_width)), mode='constant')

print("Loading extracted features...")
with open('data/processed/features_final.pkl', 'rb') as f:
    data = pickle.load(f)

print(f"Total pairs: {len(data)}")
print("Aligning pairs with DTW and padding...")

X_list = []
Y_list = []

for item in tqdm(data):
    try:
        acc_mfcc = item['accented_mfcc']
        nat_mfcc = item['native_mfcc']

        aligned_acc = align_pair(acc_mfcc, nat_mfcc)

        acc_padded = pad_or_crop(aligned_acc)
        nat_padded = pad_or_crop(nat_mfcc)

        X_list.append(acc_padded.T)
        Y_list.append(nat_padded.T)
    except Exception as e:
        continue

X = np.stack(X_list)
Y = np.stack(Y_list)

print(f"\nFinal dataset shape: X={X.shape}, Y={Y.shape}")

np.save('data/processed/X_train.npy', X)
np.save('data/processed/Y_train.npy', Y)
print("Saved aligned training data to data/processed/X_train.npy and Y_train.npy")


## 5. Training

Trains the AccentNeutralizer model on the aligned MFCC pairs.

In [ ]:
"""
train.py
Trains the AccentNeutralizer model on aligned MFCC pairs.
"""

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from neutralizer_model import AccentNeutralizer

class AccentDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

def main():
    print("Loading training data...")
    X = np.load('data/processed/X_train.npy')
    Y = np.load('data/processed/Y_train.npy')
    print(f"X shape: {X.shape}, Y shape: {Y.shape}")

    dataset = AccentDataset(X, Y)
    val_size = int(0.15 * len(dataset))
    train_size = len(dataset) - val_size
    train_ds, val_ds = random_split(dataset, [train_size, val_size])

    print(f"Train samples: {train_size}, Val samples: {val_size}")

    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    model = AccentNeutralizer().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.MSELoss()

    EPOCHS = 30
    best_val_loss = float('inf')

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * xb.size(0)
        train_loss /= train_size

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                pred = model(xb)
                loss = criterion(pred, yb)
                val_loss += loss.item() * xb.size(0)
        val_loss /= val_size

        print(f"Epoch {epoch+1}/{EPOCHS} - Train Loss: {train_loss:.4f} - Val Loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'best_model.pt')

    print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")
    print("Best model saved to best_model.pt")

if __name__ == "__main__":
    main()


## 6. Inference — Try It Out

Run the full pipeline end-to-end on a sample or your own audio file.

In [ ]:
"""
main.py
Accent Neutralizer for Speech - main entry point.

Usage:
    python main.py --input path/to/accented_audio.wav --output path/to/output.wav

If no --input is given, runs a demo using a sample from the L2-ARCTIC dataset.
"""

import argparse
import torch
import numpy as np
import librosa
import soundfile as sf
from neutralizer_model import AccentNeutralizer
from feature_extractor import FeatureExtractor

SR = 16000
MAX_FRAMES = 300


def load_model(checkpoint_path='best_model.pt'):
    device = torch.device('cpu')
    model = AccentNeutralizer().to(device)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.eval()
    return model


def neutralize_accent(audio, model, extractor):
    """Takes a raw audio waveform, returns accent-neutralized audio."""
    mfcc = extractor.extract_mfcc(audio)
    T = mfcc.shape[1]

    if T >= MAX_FRAMES:
        mfcc_input = mfcc[:, :MAX_FRAMES]
    else:
        mfcc_input = np.pad(mfcc, ((0, 0), (0, MAX_FRAMES - T)), mode='constant')

    x = torch.tensor(mfcc_input.T, dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        output = model(x)
    predicted_mfcc = output.squeeze(0).numpy().T
    predicted_mfcc = predicted_mfcc[:, :T]

    converted_audio = librosa.feature.inverse.mfcc_to_audio(
        predicted_mfcc, sr=SR, n_fft=1024, hop_length=256
    )
    return converted_audio


def run_demo():
    """Runs a demo using a sample from L2-ARCTIC (Hindi speaker) since no input file was given."""
    from datasets import load_dataset, Audio
    import io

    print("No --input provided. Running demo on a sample L2-ARCTIC (Hindi) sentence...")
    l2 = load_dataset("KoelLabs/L2Arctic")
    l2 = l2.cast_column("audio", Audio(decode=False))

    sample = None
    for row in l2['scripted']:
        if row['speaker_native_language'] == 'Hindi':
            sample = row
            break

    print(f"Demo sentence: {sample['text']}")
    audio_bytes = sample['audio']['bytes']
    audio, sr = sf.read(io.BytesIO(audio_bytes))
    return audio.astype(np.float32)


def main():
    parser = argparse.ArgumentParser(description="Accent Neutralizer for Speech")
    parser.add_argument('--input', type=str, default=None, help='Path to input accented .wav file')
    parser.add_argument('--output', type=str, default='output_neutralized.wav', help='Path to save output audio')
    parser.add_argument('--checkpoint', type=str, default='best_model.pt', help='Path to trained model checkpoint')
    args = parser.parse_args()

    print("Loading model...")
    model = load_model(args.checkpoint)
    extractor = FeatureExtractor()

    if args.input:
        print(f"Loading input audio: {args.input}")
        audio, sr = sf.read(args.input)
        audio = audio.astype(np.float32)
    else:
        audio = run_demo()

    print("Running accent neutralization...")
    converted_audio = neutralize_accent(audio, model, extractor)

    sf.write(args.output, converted_audio, SR)
    print(f"Done! Saved neutralized audio to {args.output}")


if __name__ == "__main__":
    main()


## Results (v1)

- Training loss decreased consistently over 30 epochs (621 → 350 train, 597 → 384 validation), with train/val loss tracking closely — indicating real learning without significant overfitting.
- Qualitatively, converted audio remains intelligible (sentence content is preserved) with an audible timbral shift from the source accent.

## Known Limitations

- **Audio quality**: Griffin-Lim reconstruction introduces a robotic/synthetic quality, since it estimates phase information that was discarded during MFCC extraction. This is a reconstruction-method limitation, not a failure of the conversion model itself.
- **Speaker identity preservation**: this v1 does not explicitly disentangle speaker identity from accent — some voice-identity drift is expected.
- **Prosody**: pitch/rhythm patterns are extracted but not yet incorporated into the conversion model itself.
- **Accent coverage**: limited to Hindi and Chinese non-native accents in v1.

## Future Work

- Replace Griffin-Lim with a neural vocoder (e.g., HiFi-GAN) for natural-sounding output
- Incorporate prosody (pitch/rhythm) into the conversion model
- Explicit speaker-identity preservation via separate content/speaker embeddings
- Extend to additional accent pairs (Spanish, Arabic, Korean, Vietnamese)

## Acknowledgements

- [L2-ARCTIC corpus](https://psi.engr.tamu.edu/l2-arctic-corpus/) (Texas A&M University, Iowa State University)
- [CMU ARCTIC corpus](http://festvox.org/cmu_arctic/) (Carnegie Mellon University)
